In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.formula.api as smf
from scipy.stats import pearsonr, zscore

# Task Performance Across Load Conditions

Accuracy was extracted from MATB scripts using extras/matb_point_accuracy.py. NASA-TLX scores were extracted using extras/extract_nasa_tlx.py

## Stats

### Baseline/Experimental Subtasks across conditions

In [ ]:
df = pd.read_csv("data/performance/performance_exp.csv")
df['condition'] = df['condition'].map({'L': 'Low', 'M': 'Moderate', 'H': 'High'})
df['bin_group'] = df['window_start'].astype("category")

task_cols = {
    'track_point_accuracy': 'Tracking',
    'sysmon_point_accuracy': 'System Monitoring',
    'comms_point_accuracy': 'Communications',
    'resman_point_accuracy': 'Resource Management'
}

def run_lmm(data, col, ref_level):
    data = data.copy()
    data['load_level'] = pd.Categorical(
        data['condition'],
        categories=['Low', 'Moderate', 'High'],
        ordered=True
    )
    data['load_level'] = data['load_level'].cat.reorder_categories(
        [ref_level] + [lvl for lvl in ['Low', 'Moderate', 'High'] if lvl != ref_level]
    )
    model = smf.mixedlm(
        f"{col} ~ load_level",
        data,
        groups=data["participant"],
        vc_formula={"bin_group": "0 + C(bin_group)"}
    )
    return model.fit(reml=False)

# === Sig result formatting
def format_latex(beta, p):
    beta_str = f"\\beta = {beta:.3f}"
    p_str = "p < .001" if p < 0.001 else f"p = {p:.3f}".replace("0.", ".")
    return f"${beta_str}$, ${p_str}$"


for col, label in task_cols.items():
    print(f"\n-- {label} --")
    for ref in ['Low', 'Moderate']:
        res = run_lmm(df, col, ref_level=ref)
        coefs = res.params
        pvals = res.pvalues

        print(f"(reference: {ref})")
        for term in coefs.index:
            if term.startswith("load_level"):
                beta = coefs[term]
                p = pvals[term]
                contrast = term.split('[')[-1].strip(']')
                latex_str = format_latex(beta, p)
                print(f"{contrast}: {latex_str}")


### experimental-baseline relationship

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# Load pre and main session data separately
df_pre = pd.read_csv("data/performance/performance_bsl.csv")
df_main = pd.read_csv("data/performance/performance_exp.csv")

# Participant formatting
df_pre['participant'] = df_pre['participant'].astype(str)
df_main['participant'] = df_main['participant'].astype(str)

# Task column mapping
task_cols = {
    'track_point_accuracy':  'Track',
    'resman_point_accuracy': 'ResMan',
    'sysmon_point_accuracy': 'SysMon',
    'comms_point_accuracy':  'Comms'
}

df_pre = df_pre.rename(columns=task_cols)
df_main = df_main.rename(columns=task_cols)
loads = ['H', 'M', 'L']

def make_wide(df):
    grouped = df.groupby(['participant', 'condition'])[list(task_cols.values())].mean()
    wide = grouped.unstack('condition').swaplevel(0, 1, axis=1)
    wide = wide.reindex(columns=pd.MultiIndex.from_product([loads, list(task_cols.values())]))
    return wide

def corr_with_pvalues(df1, df2=None):
    cols1 = df1.columns
    cols2 = df2.columns if df2 is not None else cols1
    r_matrix = pd.DataFrame(index=cols1, columns=cols2, dtype=float)
    p_matrix = pd.DataFrame(index=cols1, columns=cols2, dtype=float)
    for c1 in cols1:
        for c2 in cols2:
            x = df1[c1]
            y = df2[c2] if df2 is not None else df1[c2]
            mask = x.notna() & y.notna()
            r, p = pearsonr(x[mask], y[mask]) if mask.sum() > 2 else (np.nan, np.nan)
            r_matrix.loc[c1, c2] = r
            p_matrix.loc[c1, c2] = p
    return r_matrix, p_matrix

# Build matrices
base_df = make_wide(df_pre)
main_df = make_wide(df_main)

# Align participants
common = base_df.index.intersection(main_df.index)
base_df = base_df.loc[common]
main_df = main_df.loc[common]

# Correlations
base_corr, base_p = corr_with_pvalues(base_df)
main_corr, main_p = corr_with_pvalues(main_df)
cross_corr, cross_p = corr_with_pvalues(base_df, main_df)

# Save
base_corr.to_csv("data/corr_results/base_corr.csv")
base_p.to_csv("data/corr_results/base_p.csv")
main_corr.to_csv("data/corr_results/main_corr.csv")
main_p.to_csv("data/corr_results/main_p.csv")
cross_corr.to_csv("data/corr_results/cross_corr.csv")
cross_p.to_csv("data/corr_results/cross_p.csv")

print("[✓] Correlation matrices saved to data/corr_results/")


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, zscore
import statsmodels.formula.api as smf

# === Load data from separate files
df_pre = pd.read_csv("data/performance/performance_bsl.csv")
df_main = pd.read_csv("data/performance/performance_exp.csv")

# Ensure correct types
for df in [df_pre, df_main]:
    df['participant'] = df['participant'].astype(str)
    df['load_level'] = df['condition'].map({'L': 'Low', 'M': 'Moderate', 'H': 'High'})
    df['load_level'] = pd.Categorical(df['load_level'], categories=['Low', 'Moderate', 'High'], ordered=True)
    df['bin_group'] = df['window_start'].astype(str)

# === Compute participant-level means per task in pre and main
pre_agg = df_pre.groupby('participant')[
    ['track_point_accuracy', 'resman_point_accuracy', 'sysmon_point_accuracy', 'comms_point_accuracy']
].mean().add_suffix('_pre')

main_agg = df_main.groupby('participant')[
    ['track_point_accuracy', 'resman_point_accuracy', 'sysmon_point_accuracy', 'comms_point_accuracy']
].mean().add_suffix('_main')

# Merge pre and main into one DataFrame
pivoted = pd.concat([pre_agg, main_agg], axis=1).dropna()

# Compute overall accuracy
pivoted['overall_pre'] = pivoted[[c for c in pivoted.columns if c.endswith('_pre')]].mean(axis=1)
pivoted['overall_main'] = pivoted[[c for c in pivoted.columns if c.endswith('_main')]].mean(axis=1)

# === Pearson correlations
task_map = {
    'track': 'track_point_accuracy',
    'resman': 'resman_point_accuracy',
    'sysmon': 'sysmon_point_accuracy',
    'comms': 'comms_point_accuracy',
    'overall': 'overall'
}

print("\n=== Pearson Correlations (Pre vs. Main) ===")
for task, prefix in task_map.items():
    pre = f'{prefix}_pre'
    main = f'{prefix}_main'

    valid = pivoted[[pre, main]].dropna()
    zs = pd.DataFrame(zscore(valid[[pre, main]]), columns=[pre, main], index=valid.index)
    filtered = valid[(abs(zs[pre]) < 3) & (abs(zs[main]) < 3)]

    if len(filtered) >= 2:
        r, p = pearsonr(filtered[pre], filtered[main])
        print(f"{task.title()} accuracy: r = {r:.3f}, p = {p:.3f} (n = {len(filtered)})")
    else:
        print(f"{task.title()} accuracy: Not enough valid data after filtering.")

# === LMMs: Predict main accuracy from pre + load
# Merge pre-task means into main-trial data
baseline = pre_agg.reset_index()
main_df = df_main.copy()
main_df = main_df.merge(baseline, on='participant', how='left')
main_df = main_df.dropna().reset_index(drop=True)

task_pairs = {
    'track_point_accuracy': 'track_point_accuracy_pre',
    'resman_point_accuracy': 'resman_point_accuracy_pre',
    'sysmon_point_accuracy': 'sysmon_point_accuracy_pre',
    'comms_point_accuracy': 'comms_point_accuracy_pre'
}

print("\n=== Predicting Main-Session Accuracy from Baseline + Load ===")
for outcome, predictor in task_pairs.items():
    formula = f"{outcome} ~ {predictor} + load_level"
    try:
        model = smf.mixedlm(
            formula,
            main_df,
            groups=main_df["participant"],
            vc_formula={"bin_group": "0 + C(bin_group)"}
        )
        result = model.fit(reml=False)
        print(f"\n-- {outcome.replace('_point_accuracy', '').title()} --")
        print(result.summary().tables[1])
    except Exception as e:
        print(f"\n[!] Error in model for {outcome}: {e}")


## Figs

### performance bars

In [ ]:
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.2)

title_fontsize = 12
axis_label_fontsize = 11
tick_fontsize = 10
suptitle_fontsize = 14

# === Load and preprocess data
df = pd.read_csv("data/performance/performance_bsl.csv")  # Use "performance_exp.csv" if needed
df['condition'] = df['condition'].map({'L': 'Low', 'M': 'Moderate', 'H': 'High'})
df['load_level'] = pd.Categorical(df['condition'], categories=['Low', 'Moderate', 'High'], ordered=True)

task_cols = {
    'track_point_accuracy': 'Track',
    'resman_point_accuracy': 'ResMan',
    'sysmon_point_accuracy': 'SysMon',
    'comms_point_accuracy': 'Comms'
}

# === Melt and aggregate
long_df = df.melt(
    id_vars=['participant', 'load_level'],
    value_vars=list(task_cols.keys()),
    var_name='subtask', value_name='accuracy'
)
long_df['accuracy'] = pd.to_numeric(long_df['accuracy'], errors='coerce')
long_df['subtask'] = long_df['subtask'].map(task_cols)

agg_df = (
    long_df.groupby(['participant', 'load_level', 'subtask'])['accuracy']
    .mean()
    .reset_index()
)

summary = (
    agg_df.groupby(['subtask', 'load_level'])['accuracy']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
summary['mean'] *= 100
summary['sem'] = (summary['std'] / np.sqrt(summary['count'])) * 100

# === Abbreviate load levels for plotting
summary['load_level'] = summary['load_level'].map({'Low': 'L', 'Moderate': 'M', 'High': 'H'})
summary['load_level'] = pd.Categorical(summary['load_level'], categories=['L', 'M', 'H'], ordered=True)

# === Plot
condition_palette = {
    'L': '#4575b4',
    'M': '#ffffbf',
    'H': '#d73027'
}

g = sns.catplot(
    data=summary,
    kind='bar',
    x='load_level',
    y='mean',
    hue='load_level',
    col='subtask',
    errorbar=None,
    palette=condition_palette,
    height=4,
    aspect=0.9,
    legend_out=False
)

for ax, subtask in zip(g.axes.flat, summary['subtask'].unique()):
    sub = summary[summary['subtask'] == subtask]
    patches = [patch for patch in ax.patches if patch.get_height() > 0]
    for patch, (_, row) in zip(patches, sub.iterrows()):
        x = patch.get_x() + patch.get_width() / 2
        y = patch.get_height()
        err = row['sem']
        ax.errorbar(x, y, yerr=err, fmt='none', c='black', capsize=4, lw=1)

g.set_titles("{col_name}", size=title_fontsize)
g.set_ylabels("Accuracy (%)")
g.set(ylim=(40, 100))
for ax in g.axes.flat:
    ax.set_xlabel("")
    ax.tick_params(axis='x', which='both', bottom=True, top=False, length=4)
    for patch in ax.patches:
        patch.set_edgecolor('black')
        patch.set_linewidth(2.0)

g.figure.subplots_adjust(top=0.82, wspace=0.3)
g.figure.text(0.5, 0.06, 'Load Condition', ha='center', va='center', fontsize=axis_label_fontsize)


### scatter plots

In [ ]:
df_pre = pd.read_csv("data/performance/performance_bsl.csv")
df_main = pd.read_csv("data/performance/performance_exp.csv")

# Ensure consistent formatting
df_pre['participant'] = df_pre['participant'].astype(str)
df_main['participant'] = df_main['participant'].astype(str)

# Drop rows missing bin timing info (if needed)
df_pre = df_pre.dropna(subset=['window_start', 'window_end'])
df_main = df_main.dropna(subset=['window_start', 'window_end'])

# Task column mapping
task_cols = {
    'track_point_accuracy': 'Tracking',
    'resman_point_accuracy': 'ResMan',
    'sysmon_point_accuracy': 'SysMon',
    'comms_point_accuracy': 'Comms'
}

# Plot settings
title_fontsize = 11
axis_label_fontsize = 10
tick_fontsize = 9

# === Create 2x2 Grid of Scatterplots ===
fig, axes = plt.subplots(2, 2, figsize=(7, 6))  # Match Overleaf figure size
axes = axes.flatten()

for i, (col, label) in enumerate(task_cols.items()):
    # Compute participant-level means for each session
    pre_avg = df_pre.groupby('participant')[col].mean()
    main_avg = df_main.groupby('participant')[col].mean()

    # Join pre and main into single DataFrame
    pivot = pd.DataFrame({'pre': pre_avg, 'main': main_avg}).dropna()

    # Identify outliers via z-score (±3 SD)
    z = pivot.apply(zscore)
    outlier_mask = (z.abs() > 3).any(axis=1)

    # Split data
    inliers = pivot[~outlier_mask]
    outliers = pivot[outlier_mask]

    # Pearson r on inliers only
    r, _ = pearsonr(inliers['pre'], inliers['main'])

    # Plot inliers with regression line
    sns.regplot(
        x='pre', y='main', data=inliers, ax=axes[i],
        scatter_kws={'s': 40, 'color': '#d73027'},
        line_kws={'color': '#4575b4', 'lw': 3}
    )

    # Plot outliers as hollow points
    axes[i].scatter(
        outliers['pre'], outliers['main'],
        facecolors='none', edgecolors='#d73027', s=40, linewidths=1.2
    )

    axes[i].set_title(f"{label} (r = {r:.2f})", fontsize=title_fontsize)
    axes[i].tick_params(axis='both', labelsize=tick_fontsize)
    for spine in axes[i].spines.values():
        spine.set_linewidth(1.5)

# Layout adjustments
fig.tight_layout()
fig.subplots_adjust(top=0.93, wspace=0.17, hspace=0.3)

# Remove individual axis labels
for ax in axes:
    ax.set_xlabel("")
    ax.set_ylabel("")

# Add shared axis labels
fig.text(0.5, 0.04, "Baseline Session", ha='center', va='center', fontsize=11)
fig.text(0.04, 0.5, "Experimental Session", ha='center', va='center', rotation='vertical', fontsize=11)


### correlation heatmaps

In [ ]:

def format_subtask_labels(columns):
    return [task.capitalize() for _, task in columns]

def significance_mask(p_matrix, alpha=0.05):
    mask = p_matrix.copy()
    for i in range(len(mask)):
        mask.iat[i, i] = np.nan  # remove diagonal
    return mask.applymap(lambda p: "*" if pd.notnull(p) and p < alpha else "")

matrices = [
    (base_corr, base_p,  "baseline_corr_heatmap.svg"),
    (main_corr, main_p,  "experimental_corr_heatmap.svg"),
    (cross_corr, cross_p, "cross_corr_heatmap.svg")
]

for corr_matrix, p_matrix, filename in matrices:
    corr_matrix = corr_matrix[corr_matrix.columns[::-1]]
    p_matrix = p_matrix[p_matrix.columns[::-1]]

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={'aspect': 'equal'})

    sns.heatmap(
        corr_matrix,
        ax=ax,
        cmap='RdYlBu_r',
        vmin=0, vmax=1,
        linewidths=0.5, linecolor='black',
        xticklabels=format_subtask_labels(corr_matrix.columns),
        yticklabels=format_subtask_labels(corr_matrix.index),
        cbar=True,
        annot=significance_mask(p_matrix), fmt="", annot_kws={"size": 16, "weight": "bold", "color": "red"}
    )

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")
    ax.tick_params(axis='x', labelrotation=90, labelsize=12)
    ax.tick_params(axis='y', labelrotation=0,  labelsize=12)

    fig.tight_layout()
    fig.savefig(f"figs/{filename}", bbox_inches='tight')
    plt.close(fig)

    print(f"[✓] Saved: figs/{filename} (with significance markers)")

# Reaction Time

## Stats

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

# === Load and preprocess
df = pd.read_csv("data/performance/performance_exp.csv")
df['condition'] = df['condition'].map({'L': 'Low', 'M': 'Moderate', 'H': 'High'})
df['bin_group'] = df['window_start'].astype("category")
df['mean_rt'] = df[['mean_sysmon_response_time', 'mean_comms_response_time']].mean(axis=1)
df['mean_comms_response_time'] = df['mean_comms_response_time']*1000

task_cols = {
    'mean_sysmon_response_time': 'Sysmon RT',
    'mean_comms_response_time': 'Comms RT',
    'mean_rt': 'Avg RT'
}

# === LMM wrapper
def run_lmm(data, col, ref_level):
    data = data.copy()
    data['load_level'] = pd.Categorical(
        data['condition'],
        categories=['Low', 'Moderate', 'High'],
        ordered=True
    )
    data['load_level'] = data['load_level'].cat.reorder_categories(
        [ref_level] + [lvl for lvl in ['Low', 'Moderate', 'High'] if lvl != ref_level]
    )
    model = smf.mixedlm(
        f"{col} ~ load_level",
        data,
        groups=data["participant"],
        vc_formula={"bin_group": "0 + C(bin_group)"}
    )
    return model.fit(reml=False)

# === LaTeX formatting
def format_latex(beta, p):
    beta_str = f"\\beta = {beta:.3f}"
    p_str = "p < .001" if p < 0.001 else f"p = {p:.3f}".replace("0.", ".")
    return f"${beta_str}$, ${p_str}$"

# === Run models and print results
print("\n=== LMM LaTeX-formatted β and p-values ===")
for col, label in task_cols.items():
    print(f"\n-- {label} --")
    for ref in ['Low', 'Moderate']:
        try:
            res = run_lmm(df, col, ref_level=ref)
            coefs = res.params
            pvals = res.pvalues
            print(f"(reference: {ref})")
            for term in coefs.index:
                if term.startswith("load_level"):
                    beta = coefs[term]
                    p = pvals[term]
                    contrast = term.split('[')[-1].strip(']')
                    latex_str = format_latex(beta, p)
                    print(f"{contrast}: {latex_str}")

        except Exception as e:
            print(f"[!] Error fitting model for {label} (ref={ref}): {e}")


In [ ]:
import pandas as pd
from scipy.stats import pearsonr

# === Load RT data ===
main_path = "data/performance/performance_exp.csv"
pre_path = "data/performance/performance_bsl.csv"

df_main = pd.read_csv(main_path)
df_pre = pd.read_csv(pre_path)

# === Define RT variables
rt_cols = ['mean_sysmon_response_time', 'mean_comms_response_time']
rt_labels = {
    'mean_sysmon_response_time': 'System Monitoring RT',
    'mean_comms_response_time': 'Communications RT'
}

# === Compute participant-level means
pre_avg = df_pre.groupby('participant')[rt_cols].mean().reset_index()
main_avg = df_main.groupby('participant')[rt_cols].mean().reset_index()

# === Merge for correlation
merged = pd.merge(pre_avg, main_avg, on='participant', suffixes=('_pre', '_main'))

# === Correlation analysis
print("\n=== Baseline–Experimental RT Correlations ===")
for col in rt_cols:
    x = merged[f"{col}_pre"]
    y = merged[f"{col}_main"]

    # Remove outliers (±3 SD from pre)
    z = (x - x.mean()) / x.std()
    mask = z.abs() <= 3
    x, y = x[mask], y[mask]

    r, p = pearsonr(x, y)
    print(f"{rt_labels[col]}: r = {r:.3f}, p = {p:.4f}")


## Figs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# === Load and prepare data from separate files ===
df_pre  = pd.read_csv("data/performance/performance_bsl.csv")   # Baseline
df_main = pd.read_csv("data/performance/performance_exp.csv")  # Experimental

# Add session labels manually
df_pre['session']  = 'Baseline'
df_main['session'] = 'Experimental'

# Combine for plotting
df = pd.concat([df_pre, df_main], axis=0, ignore_index=True)

# Preprocessing
df['load_level'] = pd.Categorical(df['condition'], categories=['L', 'M', 'H'], ordered=True)
df['mean_comms_response_time'] = df['mean_comms_response_time'] * 1000  # Convert to ms

# === Palette for load levels ===
condition_palette = {'L': '#4575b4', 'M': '#ffffbf', 'H': '#d73027'}

# === Define RT metrics and filenames ===
rt_cols = {
    'mean_sysmon_response_time': ('Mean Sysmon RT (ms)',    'mean_response_time_barplot'),
    'mean_comms_response_time':      ('Mean Comms RT (ms)',     'mean_comms_rt_barplot')
}

# === Generate barplots ===
for col, (ylabel, fname) in rt_cols.items():
    # Summary stats
    summary = (
        df.groupby(['load_level', 'session'])[col]
          .agg(['mean', 'std', 'count'])
          .reset_index()
    )
    summary['sem'] = summary['std'] / np.sqrt(summary['count'])

    loads    = summary['load_level'].cat.categories
    sessions = ['Baseline', 'Experimental']
    x        = np.arange(len(loads))
    width    = 0.35

    fig, ax = plt.subplots(figsize=(7, 4))

    for i, sess in enumerate(sessions):
        sess_df = summary[summary['session'] == sess].set_index('load_level').reindex(loads)
        means   = sess_df['mean'].values
        sems    = sess_df['sem'].values
        pos     = x - width/2 + i*width
        colors  = [condition_palette[l] for l in loads]

        if sess == 'Baseline':
            bars = ax.bar(
                pos, means, width,
                yerr=sems, capsize=4,
                color=colors,
                edgecolor='black',
                linewidth=3,
                hatch='///',
                label=sess
            )
        else:
            bars = ax.bar(
                pos, means, width,
                yerr=sems, capsize=4,
                color=colors,
                edgecolor='black',
                linewidth=3,
                label=sess
            )

    # Labels and limits
    ax.set_xticks(x)
    ax.set_xticklabels(loads, fontsize=10)
    ax.set_xlabel('Load Level', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_ylim(bottom=0)

    # Custom legend
    #ax.legend_.remove()
    h_baseline = mpatches.Patch(
        facecolor='white', edgecolor='black', hatch='///',
        linewidth=1, label='Baseline'
    )
    h_experimental = mpatches.Patch(
        facecolor='black', edgecolor='black',
        linewidth=1, label='Experimental'
    )
    ax.legend(
        handles=[h_baseline, h_experimental],
        title='Session',
        bbox_to_anchor=(1.02, 0.5),
        loc='center left',
        frameon=False
    )


# Subjective Workload

## Stats

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import MultiComparison

path = "data/performance/aggregated_nasatlx.csv"
df = pd.read_csv(path)
df['condition'] = df['condition'].astype('category')
df['participant'] = df['participant'].astype('category')
df['condition'] = df['condition'].cat.reorder_categories(['L', 'M', 'H'], ordered=True)

tlx_vars = [
    'mental_demand',
    'physical_demand',
    'time_pressure',
    'performance',
    'effort',
    'frustration'
]

print("\n=== LMM Results: NASA-TLX ~ Condition (L is reference) ===\n")
for var in tlx_vars:
    print(f"\n--- {var.upper()} ---")
    model = smf.mixedlm(f"{var} ~ condition", df, groups=df["participant"])
    result = model.fit(reml=False)
    print(result.summary())
    print("\nPairwise contrasts (post hoc):")
    mc = MultiComparison(df[var], df['condition'])
    tbl = mc.tukeyhsd()
    print(tbl.summary())

In [ ]:
perf_path = "data/performance/performance_exp.csv"
tlx_path = "data/performance/aggregated_nasatlx.csv"

# Load and merge data
perf_df = pd.read_csv(perf_path)
perf_cols = ['comms_point_accuracy', 'resman_point_accuracy', 'sysmon_point_accuracy', 'track_point_accuracy']
perf_df['aggregated_performance'] = perf_df[perf_cols].mean(axis=1)

tlx_df = pd.read_csv(tlx_path)
merged_df = pd.merge(perf_df, tlx_df, on=['participant', 'condition'])

# TLX subscales and performance measures
tlx_subscales = [
    'mental_demand', 'physical_demand', 'time_pressure',
    'performance', 'effort', 'frustration'
]
behavioral_measures = perf_cols + ['aggregated_performance']

# Start LaTeX table string
latex_str = r"""\begin{table}[htbp]
\centering
\caption{Pearson correlations ($r$) and $p$-values between TLX subscales and task accuracy measures.}
\begin{tabular}{lcccccc}
\toprule
\textbf{Measure} & \textbf{Mental} & \textbf{Physical} & \textbf{Time Pressure} & \textbf{Performance} & \textbf{Effort} & \textbf{Frustration} \\
\midrule
"""

# Helper to format r and p
def latex_format(r, p):
    r_fmt = f"{r:.3f}"
    if p < 0.001:
        p_fmt = r"<.001"
    else:
        p_fmt = f"= {p:.3f}"
    return f"${r_fmt}$ ({p_fmt})"

# Build the table
for measure in behavioral_measures:
    row_label = measure.replace('_point_accuracy', '').replace('_', ' ').title()
    row_data = []
    for tlx in tlx_subscales:
        r, p = pearsonr(merged_df[tlx], merged_df[measure])
        row_data.append(latex_format(r, p))
    latex_str += row_label + " & " + " & ".join(row_data) + r" \\" + "\n"

# Finish the LaTeX table
latex_str += r"""\bottomrule
\end{tabular}
\end{table}
"""

# Output the LaTeX table string
print(latex_str)
